### trial

In [ ]:
import pybullet as p
import time
import pybullet_data

physicsClient=p.connect(p.GUI)

p.setAdditionalSearchPath(pybullet_data.getDataPath())

p.setGravity(0,0,-91.81)

startpos=[0,0,1]

startOrientation=p.getQuaternionFromEuler([0,0,0])

planeId=p.loadURDF("plane.urdf")

robotId=p.loadURDF("r2d2.urdf",startpos, startOrientation)


numJoints = p.getNumJoints(robotId)

print("Number of joints:", numJoints)

for i in range (numJoints):
    joint_info=p.getJointInfo(robotId, i)
    joint_id=joint_info[0]
    joint_name=joint_info[1].decode('utf-8')
    joint_type=joint_info[2]
    print(f"Joint {i}: ID={joint_id}, Name={joint_name}, Type:{joint_type}")



# try:
#     while True:
#         p.stepSimulation()
#         time.sleep(1./240.)
# except KeyboardInterrupt:
#     pass

p.disconnect()







### playing around

In [ ]:
import pybullet as p
import time
import pybullet_data

p.connect(p.GUI)

p.setAdditionalSearchPath(pybullet_data.getDataPath())

p.setGravity(0,0,-91.81)
startpos=[0,0,1]
start_oriantation=p.getQuaternionFromEuler([0,0,0])
planeId=p.loadURDF("plane.urdf")
robotId=p.loadURDF("r2d2.urdf",startpos, start_oriantation)

head_slider=p.addUserDebugParameter("r2d2 head yaw", -3.14, 3.14, 0)

try:
    while True:

        target_pos=p.readUserDebugParameter(head_slider)

        p.setJointMotorControl2(robotId,jointIndex=13,controlMode=p.POSITION_CONTROL
                                ,targetPosition=target_pos,force=50)

        p.stepSimulation()
        time.sleep(1/240.)

except KeyboardInterrupt:
    pass
p.disconnect()


### ik

In [ ]:
import pybullet as p
import time
import pybullet_data

p.connect(p.GUI)

p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0,0,-9.81)

planeId = p.loadURDF("plane.urdf")

arm_id=p.loadURDF("kuka_iiwa/model.urdf", [0.0, 0.0, 0.0], useFixedBase=True)

num_joints = p.getNumJoints(arm_id)
for i in range(num_joints):
    joint_info=p.getJointInfo(arm_id, i)
    joint_id=joint_info[0]
    joint_name=joint_info[1].decode('utf-8')
    joint_type=joint_info[2]
    print(f"Joint {i}: ID={joint_id}, Name={joint_name}, Type:{joint_type}")

x_slider=p.addUserDebugParameter("target x", -0.8, 0.8, 0)
y_slider=p.addUserDebugParameter("target y", -0.8, 0.8, 0)
z_slider=p.addUserDebugParameter("target z", 0, 1.5, 0.5)

try:
    while True:
        target_x=p.readUserDebugParameter(x_slider)
        target_y=p.readUserDebugParameter(y_slider)
        target_z=p.readUserDebugParameter(z_slider)

        target_pos=[target_x, target_y, target_z]

        joint_angles=p.calculateInverseKinematics(arm_id, 6, target_pos)

        for i in range(num_joints):
            p.setJointMotorControl2(arm_id, jointIndex=i, controlMode=p.POSITION_CONTROL,
                                    targetPosition=joint_angles[i],force=100)
            
        p.addUserDebugLine(target_pos,[target_pos[0], target_pos[1], target_pos[2]+0.1], [1,0,0], 2, 0.1)
        p.stepSimulation()
        time.sleep(1/240.)
except KeyboardInterrupt:
    pass

p.disconnect()

### lets attach a cam

In [ ]:
import pybullet as p
import time
import pybullet_data

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0,0,-9.81)
p.loadURDF("plane.urdf")
p.loadURDF("kuka_iiwa/model.urdf", [0,0,0], useFixedBase=True)


width=640
height=480
fov=60
aspect=width/height
near=0.1
far=5

lense_matrix=p.computeProjectionMatrixFOV(fov, aspect, near, far)

cam_pos=[1.0,0.5,1.0]
cam_lookat=[0,0,0.5]
cam_up=[0,0,1]

view_matrix=p.computeViewMatrix(cam_pos, cam_lookat, cam_up)

for i in range(1000):
    p.stepSimulation()

print("Taking picture...")


image_width,image_height,rgb_pixels,depth_pixels,segmentation_mask=p.getCameraImage(
    width,height,view_matrix,lense_matrix
)

# print("RGB Pixels shape:", rgb_pixels.shape)
# print("Depth Pixels shape:", depth_pixels.shape)

time.sleep(2)
p.disconnect()




### eye in hand 

In [ ]:
import pybullet as p
import time
import pybullet_data
import numpy as np
import cv2


p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0,0,-9.81)
plane_id=p.loadURDF("plane.urdf")
arm_id=p.loadURDF("kuka_iiwa/model.urdf", [0,0,0], useFixedBase=True) # why did i pick this robot ?? i kew they had a medical robot :)

num_joints = p.getNumJoints(arm_id)

end_effector_index=6 # searched in goofle for kuka iiwa end effector index

width,height=640,640
projection_matrix=p.computeProjectionMatrixFOV(fov=60, aspect=width/height, nearVal=0.1, farVal=5)

x_slider=p.addUserDebugParameter("target x", -0.8, 0.8, 0)
y_slider=p.addUserDebugParameter("target y", -0.8, 0.8, 0)
z_slider=p.addUserDebugParameter("target z", 0, 1.5, 0)
# abovee is the reach envolope of the robot why ths ?? just to test i have kept it low idk how much is this :) 

try:
    while True:

        target_pos=[p.readUserDebugParameter(x_slider),
                    p.readUserDebugParameter(y_slider),
                    p.readUserDebugParameter(z_slider)]
        
        joint_angles=p.calculateInverseKinematics(arm_id, end_effector_index, target_pos)

        for i in range(num_joints):
            p.setJointMotorControl2(arm_id, jointIndex=i, controlMode=p.POSITION_CONTROL,
                                    targetPosition=joint_angles[i],force=100)
            
        gripper_state=p.getLinkState(arm_id, end_effector_index,computeForwardKinematics=True)
        gripper_pos=gripper_state[0]
        gripper_orientation=gripper_state[1] #x,y,z,w
        gripper_rotation_matrix=p.getMatrixFromQuaternion(gripper_orientation) 
        gripper_rotation_matrix=np.array(gripper_rotation_matrix).reshape(3,3)

        cam_forward_vector=gripper_rotation_matrix.dot([0,0,1]) #z axis of the end effector here z is forword and x is right and y is up
        #but we to change it to z as up and x as forward and y as right for the camera so we will swap the axes

        cam_up_vector=gripper_rotation_matrix.dot([0,1,0])# similarly y axis of the end effector is up for the camera

        camera_eye=gripper_pos
        camera_target=gripper_pos+cam_forward_vector*1.0 # looking 1 m ahead of the end effector

        view_matrix=p.computeViewMatrix(camera_eye, camera_target, cam_up_vector)

        image_width,image_height,rgb_pixels,depth_pixels,segmentation_mask=p.getCameraImage(
            width,height,view_matrix,projection_matrix
        )

        rgb_array=np.reshape(rgb_pixels, (image_height, image_width, 4)).astype(np.uint8)[...,:3] # we only need rgb channels not alpha
        bgr=cv2.cvtColor(rgb_array, cv2.COLOR_RGB2BGR)
        cv2.imshow("Camera View", bgr)
        key=cv2.waitKey(1) & 0xFF
        if key==ord('q'):
            break   

        p.stepSimulation()
        time.sleep(1/240.)
except KeyboardInterrupt:
    pass
cv2.destroyAllWindows()

p.disconnect()




### pick a object

In [ ]:
import pybullet as p
import time
import pybullet_data

p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0,0,-9.81)
plane_id=p.loadURDF("plane.urdf")


robotId=p.loadURDF("franka_panda/panda.urdf", [0,0,0], useFixedBase=True)
end_effector_index=11 # searched in goofle for panda end effector index
num_joints = p.getNumJoints(robotId)

block_id=p.loadURDF("cube_small.urdf", [0.5,0,0.5])

p.changeDynamics(block_id, -1, lateralFriction=2.0, spinningFriction=1.0, rollingFriction=1.0)
# p.changeVisualShape(block_id, -1, rgbaColor=[1,0,0,1]) # make the block red
p.changeDynamics(robotId, 9, lateralFriction=2.0, spinningFriction=1.0, rollingFriction=1.0)
p.changeDynamics(robotId, end_effector_index, lateralFriction=2.0, spinningFriction=1.0, rollingFriction=1.0)

x_slider=p.addUserDebugParameter("target x", -0.8, 0.8, 0) #0.5
y_slider=p.addUserDebugParameter("target y", -0.8, 0.8, 0)
z_slider=p.addUserDebugParameter("target z", 0, 1.5, 0) #0.3

gripper_slider=p.addUserDebugParameter("gripper open", 0, 1, 0)



try:
    while True:

        target_pos=[p.readUserDebugParameter(x_slider),
        p.readUserDebugParameter(y_slider), 
        p.readUserDebugParameter(z_slider)]

        down_oriantation=p.getQuaternionFromEuler([3.14,0,0]) # gripper facing down 180,0,0 in euler angles 
        # we need to add this because ik will be correct 

        joint_angles=p.calculateInverseKinematics(robotId, end_effector_index, target_pos, down_oriantation)

        for i in range(7): # only control the 7 joints of the arm not the gripper (if not tuple index out of range error will come )
            p.setJointMotorControl2(robotId, jointIndex=i, controlMode=p.POSITION_CONTROL,
                                    targetPosition=joint_angles[i],force=100)
            
        gripper_opening=p.readUserDebugParameter(gripper_slider)
        p.setJointMotorControl2(robotId, 9, controlMode=p.POSITION_CONTROL, targetPosition=gripper_opening*0.04, force=50)
        p.setJointMotorControl2(robotId, 10, controlMode=p.POSITION_CONTROL, targetPosition=gripper_opening*0.04, force=50)
        #gemini suggested me to add this why because we need to reduce  force 
        p.stepSimulation()
        time.sleep(1/240.)

except KeyboardInterrupt:
    pass
p.disconnect()


### state managenment 

In [ ]:
import pybullet as p
import pybullet_data
import time

# 1. Setup Environment
p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)
p.loadURDF("plane.urdf")

# 2. Load Arm and Block
robotId = p.loadURDF("franka_panda/panda.urdf", [0, 0, 0], useFixedBase=True)
endEffectorIndex = 11 
blockId = p.loadURDF("cube_small.urdf", [0.5, 0, 0.05])

# 3. State Machine Variables
state = 0
state_timer = 0.0
dt = 1./240.

# We will store our "Virtual Magnet" ID here
cid = None 

target_pos = [0.5, 0.0, 0.3] 
gripper_target = 0.04        
down_orientation = p.getQuaternionFromEuler([3.14, 0, 0])

print("Initiating Bulletproof Pick-and-Place Sequence...")

try:
    while True:
        state_timer += dt 
        
        if state == 0: # Hover
            target_pos = [0.5, 0.0, 0.3]
            gripper_target = 0.04
            if state_timer > 2.0: 
                state = 1
                state_timer = 0.0 
                print("State 1: Reaching down")
                
        elif state == 1: # Reach down
            target_pos = [0.5, 0.0, 0.11] # Down to the block
            if state_timer > 2.0:
                state = 2
                state_timer = 0.0
                print("State 2: Engaging Virtual Magnet")
                
        elif state == 2: # Grasp using Constraint
            gripper_target = 0.0 # Close fingers visually
            
            # THE CHEAT CODE: If the magnet doesn't exist yet, create it!
            if cid is None:
                # This glues the robot's end-effector (link 11) to the block (link -1)
                cid = p.createConstraint(parentBodyUniqueId=robotId, 
                                         parentLinkIndex=endEffectorIndex, 
                                         childBodyUniqueId=blockId, 
                                         childLinkIndex=-1, 
                                         jointType=p.JOINT_FIXED, 
                                         jointAxis=[0, 0, 0], 
                                         parentFramePosition=[0, 0, 0.05], # Offset so it sits in the fingers
                                         childFramePosition=[0, 0, 0])
            if state_timer > 1.0:
                state = 3
                state_timer = 0.0
                print("State 3: Lifting")
                
        elif state == 3: # Lift
            target_pos = [0.5, 0.0, 0.4] 
            if state_timer > 2.0:
                state = 4
                state_timer = 0.0
                print("State 4: Moving to drop zone")
                
        elif state == 4: # Move to drop zone
            target_pos = [0.5, 0.4, 0.4] 
            if state_timer > 2.0:
                state = 5
                state_timer = 0.0
                print("State 5: Releasing Magnet")
                
        elif state == 5: # Release
            gripper_target = 0.04 # Open fingers visually
            
            # Remove the constraint to drop the block
            if cid is not None:
                p.removeConstraint(cid)
                cid = None # Reset it
                
            if state_timer > 2.0:
                state = 6
                print("Sequence Complete!")

        # --- EXECUTE COMMANDS ---
        joint_angles = p.calculateInverseKinematics(robotId, endEffectorIndex, target_pos, targetOrientation=down_orientation)
        
        for i in range(7):
            p.setJointMotorControl2(robotId, i, p.POSITION_CONTROL, joint_angles[i], force=100)
            
        p.setJointMotorControl2(robotId, 9, p.POSITION_CONTROL, gripper_target, force=50)
        p.setJointMotorControl2(robotId, 10, p.POSITION_CONTROL, gripper_target, force=50)

        p.stepSimulation()
        time.sleep(dt)

except KeyboardInterrupt:
    pass

p.disconnect()

In [ ]:
import pybullet as p
import pybullet_data
import time
import numpy as np
import cv2

# ---------------------------------------------------------
# HELPER FUNCTION: Convert 2D Pixel + Depth to 3D World Coordinate
# ---------------------------------------------------------
def get_3d_position(pixel_x, pixel_y, depth_image, view_matrix, proj_matrix, width, height):
    # 1. Get the depth value [0.0 to 1.0] at the pixel
    depth_val = depth_image[pixel_y, pixel_x]

    # 2. Convert pixel coordinates to Normalized Device Coordinates (NDC) [-1 to 1]
    ndc_x = (2.0 * pixel_x / width) - 1.0
    ndc_y = 1.0 - (2.0 * pixel_y / height) # Y is inverted in images
    ndc_z = 2.0 * depth_val - 1.0

    # 3. Create the inverse View-Projection matrix
    view_matrix_np = np.array(view_matrix).reshape(4, 4).T
    proj_matrix_np = np.array(proj_matrix).reshape(4, 4).T
    vp_matrix = np.dot(proj_matrix_np, view_matrix_np)
    inv_vp_matrix = np.linalg.inv(vp_matrix)

    # 4. Multiply NDC by inverse matrix to un-project
    clip_space_pos = np.array([ndc_x, ndc_y, ndc_z, 1.0])
    world_pos = np.dot(inv_vp_matrix, clip_space_pos)

    # 5. Normalize by the W coordinate
    world_pos /= world_pos[3]
    return world_pos[0:3] # Return X, Y, Z

# ---------------------------------------------------------
# MAIN SIMULATION
# ---------------------------------------------------------
p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.loadURDF("plane.urdf")

# Load Panda Arm and a RED block at a random location
robotId = p.loadURDF("franka_panda/panda.urdf", [0, 0, 0], useFixedBase=True)

# Spawning a block somewhere in front of the robot
# (Change these numbers to see the vision system adapt!)
block_true_pos = [0.6, 0.2, 0.05] 
blockId = p.loadURDF("cube_small.urdf", block_true_pos)
p.changeVisualShape(blockId, -1, rgbaColor=[1, 0, 0, 1]) # Make it Bright Red

# --- Setup Hand-in-Eye (Static) Camera ---
width, height = 640, 480
fov, aspect, near, far = 60, width/height, 0.02, 5.0
proj_matrix = p.computeProjectionMatrixFOV(fov, aspect, near, far)

# Place camera up high, looking down at the workspace
cam_eye = [0.5, 0.0, 1.0]
cam_target = [0.5, 0.0, 0.0]
cam_up = [1, 0, 0]
view_matrix = p.computeViewMatrix(cam_eye, cam_target, cam_up)

# Let physics settle
for _ in range(50): p.stepSimulation()

print("Taking a picture and processing vision...")

# 1. Capture Image
_, _, rgb_pixels, depth_pixels, _ = p.getCameraImage(width, height, view_matrix, proj_matrix)

# 2. OpenCV Processing
rgb_array = np.reshape(rgb_pixels, (height, width, 4)).astype(np.uint8)[...,:3] # Convert to BGR and drop alpha
bgr_image = cv2.cvtColor(rgb_array, cv2.COLOR_RGBA2BGR)
hsv_image = cv2.cvtColor(bgr_image, cv2.COLOR_BGR2HSV)

# Define range for Red color in HSV
lower_red = np.array([0, 120, 70])
upper_red = np.array([10, 255, 255])
mask = cv2.inRange(hsv_image, lower_red, upper_red)

# Find contours (blobs) of the red object
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

if contours:
    # Get the largest red blob
    largest_contour = max(contours, key=cv2.contourArea)
    
    # Calculate the center pixel (u, v) using image moments
    M = cv2.moments(largest_contour)
    if M["m00"] != 0:
        pixel_x = int(M["m10"] / M["m00"]) # 'u' coordinate
        pixel_y = int(M["m01"] / M["m00"]) # 'v' coordinate
        
        print(f"Object found at pixel: X={pixel_x}, Y={pixel_y}")
        
        # 3. Convert 2D pixel to 3D World Coordinate!
        depth_image = np.reshape(depth_pixels, (height, width))
        target_3d = get_3d_position(pixel_x, pixel_y, depth_image, view_matrix, proj_matrix, width, height)
        
        print(f"Calculated 3D Position: X={target_3d[0]:.3f}, Y={target_3d[1]:.3f}, Z={target_3d[2]:.3f}")
        print(f"Actual Ground Truth:    X={block_true_pos[0]:.3f}, Y={block_true_pos[1]:.3f}, Z={block_true_pos[2]:.3f}")

        # Draw a circle on the image to prove we found it
        cv2.circle(bgr_image, (pixel_x, pixel_y), 5, (0, 255, 0), -1)

cv2.imshow("Robot Vision", bgr_image)
cv2.imshow("Red Mask", mask)
cv2.waitKey(0) # Press any key on the image windows to close them

p.disconnect()

Taking a picture and processing vision...
Object found at pixel: X=232, Y=194
Calculated 3D Position: X=0.602, Y=0.196, Z=0.075
Actual Ground Truth:    X=0.600, Y=0.200, Z=0.050
